## **Step 1: Imports**

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

## **Step 2: Load Dataset**

In [ ]:
df = pd.read_csv(
    "soc_analyst_dataset_200k.csv"
)

print(df.shape)

(200000, 14)


## **Step 3: Prepare Target**

Target:

In [ ]:
target = "priority_label"

Encode:

In [ ]:
le = LabelEncoder()

df[target] = le.fit_transform(
    df[target]
)

## **Step 4: Feature Selection**

Drop labels

In [ ]:
drop_cols = [
    "alert_id",
    "priority_label",
    "response_action"
]

In [ ]:
X = df.drop(
    columns=drop_cols
)

y = df[target]

## **Step 5: Categorical/Numerical Split**

In [ ]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

## **Step 6: Preprocessing**

Numeric:

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

Categorical:

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

## **Step 7: Column Transformer**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

## **Step 8: Split Data**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## **Step 9: Model**

In [ ]:
priority_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    random_state=42,
    n_jobs=-1
)

## **Step 10: Build Pipeline**

In [ ]:
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            priority_model
        )
    ]
)

## **Step 11: Train**

In [ ]:
pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['severity_raw',
                                                   'failed_login_count',
                                                   'geo_anomaly',
                                                   'known_ioc_match',
                                                   'malware_detected',
                                                   'user_risk_score',
                                                   'overall_risk_score']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['alert_source', 'alert_type',
                                                   'asset_criticality',
                                                   'threat_label'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=18, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

## **Step 12: Evaluate**

In [ ]:
y_pred = pipeline.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       463
           1       1.00      1.00      1.00      2370
           2       1.00      1.00      1.00     10054
           3       1.00      1.00      1.00     27113

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000

[[  463     0     0     0]
 [    0  2370     0     0]
 [    0     0 10054     0]
 [    0     0     0 27113]]


## **Step 13: Save Model**

In [ ]:
joblib.dump(
    pipeline,
    "priority_model.pkl"
)

['priority_model.pkl']

## **Step 14: Save Label Encoder**

In [ ]:
joblib.dump(
    le,
    "priority_encoder.pkl"
)

['priority_encoder.pkl']

## **Step 15: Verify**

In [ ]:
import os

print(
    os.path.exists(
        "priority_model.pkl"
    )
)

print(
    os.path.exists(
        "priority_encoder.pkl"
    )
)

True
True


## **Step 16: Download**

In [ ]:
from google.colab import files

files.download(
    "priority_model.pkl"
)

files.download(
    "priority_encoder.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Test Prediction**

In [ ]:
sample = X_test.iloc[[0]]

pred = pipeline.predict(sample)

priority = le.inverse_transform(pred)

print(priority)

['P4']


In [ ]:
print(X.columns.tolist())

['alert_source', 'alert_type', 'severity_raw', 'failed_login_count', 'geo_anomaly', 'known_ioc_match', 'malware_detected', 'asset_criticality', 'user_risk_score', 'overall_risk_score', 'threat_label']
